This implemenation is Also applicable for , pinecone , FASSI etc

In [5]:
import sys
print(sys.executable)


/media/ayush-paliwal/New Volume2/LangChain Models/venv/bin/python


In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

In [7]:
from langchain_core.documents import Document


In [8]:
doc1 = Document(
    page_content="Machine Learning is a subset of Artificial Intelligence that enables systems to learn from data without being explicitly programmed.",
    metadata={"source": "ml_notes", "topic": "Machine Learning"}
)

doc2 = Document(
    page_content="Deep Learning is a branch of Machine Learning that uses neural networks with multiple layers to model complex patterns in data.",
    metadata={"source": "dl_notes", "topic": "Deep Learning"}
)

doc3 = Document(
    page_content="Natural Language Processing is a field of AI that focuses on enabling computers to understand, interpret, and generate human language.",
    metadata={"source": "nlp_notes", "topic": "NLP"}
)

doc4 = Document(
    page_content="Retrieval Augmented Generation combines information retrieval with large language models to provide more accurate and up-to-date responses.",
    metadata={"source": "rag_notes", "topic": "RAG"}
)

doc5 = Document(
    page_content="Vector databases store embeddings and allow similarity search, making them essential components of modern RAG systems.",
    metadata={"source": "vector_db_notes", "topic": "Vector Databases"}
)

In [9]:
documents = [doc1 , doc2 , doc3 , doc4 , doc5]

In [16]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

In [17]:
vectorStore = Chroma(
    embedding_function = embeddings,
    persist_directory = 'my_chroma_db', # folder where data (embedding vectors will store)
    collection_name ="sample"
)

In [ ]:
#  add Documents in vectore store:  add_documents()

vectorStore.add_documents(documents) # every doc has it own id ..

['a1c6770b-af2d-40d4-a9b4-6c55a8826401',
 'd834f8e7-9477-4a7a-8ab5-65f352da5745',
 '7549c2ec-0ce0-466c-b035-c81888321d18',
 'a2d0b9c4-37cb-4af3-8529-3ef610240576',
 '4d024bc1-2c39-4661-baa2-821c66c9d22b']

In [35]:
# view documents : get()

data = vectorStore.get(
    include=["embeddings", "documents", "metadatas"]
)

print(data)

{'ids': ['a1c6770b-af2d-40d4-a9b4-6c55a8826401', 'd834f8e7-9477-4a7a-8ab5-65f352da5745', '7549c2ec-0ce0-466c-b035-c81888321d18', 'a2d0b9c4-37cb-4af3-8529-3ef610240576', '4d024bc1-2c39-4661-baa2-821c66c9d22b'], 'embeddings': array([[-1.60559639e-02,  8.39733146e-03, -4.93864715e-03, ...,
         5.06287394e-03,  1.14987837e-02, -1.20988227e-02],
       [-9.82235093e-03,  1.20540122e-02, -1.46081415e-03, ...,
         7.23985070e-03, -6.72515668e-03, -1.62233375e-02],
       [ 1.13155541e-03,  7.32822903e-03, -1.28508254e-03, ...,
         1.12153345e-03,  5.84745954e-04, -7.19384197e-03],
       [-2.61221845e-02,  1.91694666e-02, -4.32336470e-03, ...,
         1.42826035e-03,  1.99503172e-03,  8.97652376e-03],
       [-3.24120112e-02, -2.53631206e-05, -1.64865851e-02, ...,
         8.37005489e-03,  1.03052286e-02, -2.26600026e-03]],
      shape=(5, 3072)), 'documents': ['Machine Learning is a subset of Artificial Intelligence that enables systems to learn from data without being explic

In [23]:
# search Document : similarity_search()

vectorStore.similarity_search(
    query='What is used for communication with machines?',
    k = 2
)

[Document(id='7549c2ec-0ce0-466c-b035-c81888321d18', metadata={'source': 'nlp_notes', 'topic': 'NLP'}, page_content='Natural Language Processing is a field of AI that focuses on enabling computers to understand, interpret, and generate human language.'),
 Document(id='a1c6770b-af2d-40d4-a9b4-6c55a8826401', metadata={'topic': 'Machine Learning', 'source': 'ml_notes'}, page_content='Machine Learning is a subset of Artificial Intelligence that enables systems to learn from data without being explicitly programmed.')]

In [24]:
# search Document with similarity score : similarity_search_with_score()

vectorStore.similarity_search_with_score(
    query='What is used for communication with machines?',
    k = 2
)

[(Document(id='7549c2ec-0ce0-466c-b035-c81888321d18', metadata={'source': 'nlp_notes', 'topic': 'NLP'}, page_content='Natural Language Processing is a field of AI that focuses on enabling computers to understand, interpret, and generate human language.'),
  0.6377695798873901),
 (Document(id='a1c6770b-af2d-40d4-a9b4-6c55a8826401', metadata={'source': 'ml_notes', 'topic': 'Machine Learning'}, page_content='Machine Learning is a subset of Artificial Intelligence that enables systems to learn from data without being explicitly programmed.'),
  0.6514907479286194)]

In [30]:
# meta data filtering :

results = vectorStore.similarity_search_with_score(
    query="What is NLP?",
    filter={"source": "nlp_notes"},
    k=2
)

for doc, score in results:
    print(doc.page_content)
    print(score)

Natural Language Processing is a field of AI that focuses on enabling computers to understand, interpret, and generate human language.
0.4930541515350342


In [33]:
# Update document :


updated_doc = Document(
    page_content="""
    I am updating last One because this is better version with more information .
    Natural Language Processing (NLP) is a branch of Artificial Intelligence
    that enables computers to understand, interpret, generate, and analyze
    human language. NLP combines techniques from linguistics, machine learning,
    and deep learning to perform tasks such as sentiment analysis, machine
    translation, text summarization, question answering, and chatbots.
    """,
    metadata={
        "source": "nlp_notes",
        "topic": "NLP"
    }
)

vectorStore.update_document(document_id="7549c2ec-0ce0-466c-b035-c81888321d18" , document=updated_doc)

In [36]:
data = vectorStore.get(
    include=["embeddings", "documents", "metadatas"]
)

print(data)

{'ids': ['a1c6770b-af2d-40d4-a9b4-6c55a8826401', 'd834f8e7-9477-4a7a-8ab5-65f352da5745', '7549c2ec-0ce0-466c-b035-c81888321d18', 'a2d0b9c4-37cb-4af3-8529-3ef610240576', '4d024bc1-2c39-4661-baa2-821c66c9d22b'], 'embeddings': array([[-1.60559639e-02,  8.39733146e-03, -4.93864715e-03, ...,
         5.06287394e-03,  1.14987837e-02, -1.20988227e-02],
       [-9.82235093e-03,  1.20540122e-02, -1.46081415e-03, ...,
         7.23985070e-03, -6.72515668e-03, -1.62233375e-02],
       [ 1.13155541e-03,  7.32822903e-03, -1.28508254e-03, ...,
         1.12153345e-03,  5.84745954e-04, -7.19384197e-03],
       [-2.61221845e-02,  1.91694666e-02, -4.32336470e-03, ...,
         1.42826035e-03,  1.99503172e-03,  8.97652376e-03],
       [-3.24120112e-02, -2.53631206e-05, -1.64865851e-02, ...,
         8.37005489e-03,  1.03052286e-02, -2.26600026e-03]],
      shape=(5, 3072)), 'documents': ['Machine Learning is a subset of Artificial Intelligence that enables systems to learn from data without being explic

In [37]:
# delete document : 

vectorStore.delete(ids=["7549c2ec-0ce0-466c-b035-c81888321d18"])

In [39]:
vectorStore.get(
    include=["embeddings", "documents", "metadatas"]
)


{'ids': ['a1c6770b-af2d-40d4-a9b4-6c55a8826401',
  'd834f8e7-9477-4a7a-8ab5-65f352da5745',
  'a2d0b9c4-37cb-4af3-8529-3ef610240576',
  '4d024bc1-2c39-4661-baa2-821c66c9d22b'],
 'embeddings': array([[-1.60559639e-02,  8.39733146e-03, -4.93864715e-03, ...,
          5.06287394e-03,  1.14987837e-02, -1.20988227e-02],
        [-9.82235093e-03,  1.20540122e-02, -1.46081415e-03, ...,
          7.23985070e-03, -6.72515668e-03, -1.62233375e-02],
        [-2.61221845e-02,  1.91694666e-02, -4.32336470e-03, ...,
          1.42826035e-03,  1.99503172e-03,  8.97652376e-03],
        [-3.24120112e-02, -2.53631206e-05, -1.64865851e-02, ...,
          8.37005489e-03,  1.03052286e-02, -2.26600026e-03]],
       shape=(4, 3072)),
 'documents': ['Machine Learning is a subset of Artificial Intelligence that enables systems to learn from data without being explicitly programmed.',
  'Deep Learning is a branch of Machine Learning that uses neural networks with multiple layers to model complex patterns in data